In [4]:
import pandas as pd

df_ground_truth = pd.read_csv("/workspaces/llm-zoomcamp-2026-code/04-evaluation/data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [5]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [6]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [7]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [8]:
q = ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [9]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [11]:
from google import genai
from google.genai.types import HttpOptions

client = genai.Client(
    vertexai=True,
    project="llm-zoomcamp-500505",
    location="us-central1",
    http_options=HttpOptions(api_version="v1"),
)

In [18]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=client,
    course="llm-zoomcamp",
    model="gemini-2.5-flash",
)

In [14]:
q['question']

'How do I join the Office Hours or live workshop if I don’t have the Zoom link?'

In [19]:
print(assistant.model)

gemini-2.5-flash


In [20]:
answer = assistant.rag(q['question'])

In [ ]:
assistant.total_cost()

0.0018675

In [21]:
print(answer)

You won't receive a Zoom link for Office Hours or live workshop sessions, as these are only published to instructors, presenters, and TAs.

As a student, you can participate via:
*   **YouTube Live**: Watch the live stream. The video URL will be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before the session begins. You can also find it directly on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).
*   **Slido**: Submit your questions via Slido, whose link will be pinned in the chat during the live session.


In [22]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [23]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': "You won't receive a Zoom link for Office Hours or live workshop sessions, as these are only published to instructors, presenters, and TAs.\n\nAs a student, you can participate via:\n*   **YouTube Live**: Watch the live stream. The video URL will be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before the session begins. You can also find it directly on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n*   **Slido**: Submit your questions via Slido, whose link will be pinned in the chat during the live session.",
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) bef

In [24]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [25]:
record = generate_rag_answer(q)
record

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': 'The Zoom link for Office Hours or live workshop sessions is only published to instructors, presenters, and TAs.\n\nAs a student, you can participate via YouTube Live. The video URL will be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before the session begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nYou can submit questions via Slido, and the link for Slido will be pinned in the chat when the session is live.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [Yo

In [26]:
assistant.total_cost()

0.002895

In [27]:
assistant.reset_usage()

In [28]:
assistant.total_cost()

0.0

In [29]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [31]:
with ThreadPoolExecutor(max_workers=1) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/395 [00:00<?, ?it/s]

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}

In [ ]:
results[:10]

[{'question': 'Is it okay to join the course late if I just found it now?',
  'answer_llm': 'Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'answer_llm': 'Yes, you can still join if you missed the start date, but if you want a certificate, you need to finish with the live cohort and submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'answer_llm': 'Yes, as long 

In [ ]:
df_results = pd.DataFrame(results)

In [ ]:
df_results.head()

,question,answer_llm,answer_orig,document
0,Is it okay to join the course late if I just f...,"Yes, you can still join the course late. If yo...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still take this course even if I missed ...,"Yes, you can still join if you missed the star...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,If I join after the course has already started...,"Yes, as long as you join while the course is s...","Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit my project before submissi...,"Yes — to get the certificate, you need to subm...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,I’m a bit late to the course—what do I need to...,"To still earn the certificate, you need to:\n\...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [ ]:
assistant.total_cost()


0.34332825

In [ ]:
df_results.to_csv("data/rag-answers-new.csv", index=False)